# Анализ партий на Lichess

In [ ]:
import io
from datetime import datetime, timedelta

import chess.pgn
import matplotlib.pyplot as plt
import pandas as pd
import requests

In [ ]:
username = 'AndrewSalmin'

WEEKDAY_NAMES_SHORT_RU = ['пн', 'вт', 'ср', 'чт', 'пт', 'сб', 'вс']
WEEKDAY_NAMES_FULL_RU = [
    'понедельник', 'вторник', 'среда', 'четверг',
    'пятница', 'суббота', 'воскресенье',
]

RESULT_ORDER = ['win', 'loss', 'draw', 'other']
RESULT_LABELS = {'win': 'победа', 'loss': 'поражение', 'draw': 'ничья', 'other': 'без результата'}
RESULT_COLORS = {'win': 'tab:green', 'loss': 'tab:red', 'draw': 'tab:gray', 'other': 'tab:orange'}

In [ ]:
def download_pgn_games(username: str) -> str:
    url = f'https://lichess.org/api/games/user/{username}'
    # без User-Agent Lichess отдаёт HTML-заглушку (антибот) вместо API-ответа
    headers = {'User-Agent': 'chess-analysis (github.com/andrewsalmin/chess-analysis)'}
    response = requests.get(url, headers=headers, timeout=60)

    if response.status_code == 404:
        raise ValueError(f'Пользователь "{username}" не найден на Lichess')
    response.raise_for_status()

    return response.text

pgn_games_string = download_pgn_games(username)

assert pgn_games_string, 'Пользователь не сыграл ни одной партии'

In [ ]:
pgn_games = io.StringIO(pgn_games_string)
games = []
while True:
    game = chess.pgn.read_game(pgn_games)
    if game is None:
        break
    games.append(game)

print(f'Прочитано партий: {len(games)}')

## Подготовка данных

In [ ]:
def my_result(white: str, black: str, result: str) -> str:
    if result == '1/2-1/2':
        return 'draw'
    if (white == username and result == '1-0') or (black == username and result == '0-1'):
        return 'win'
    if (white == username and result == '0-1') or (black == username and result == '1-0'):
        return 'loss'
    return 'other'  # партия прервана/отменена, результата нет


rows = []
for game in games:
    headers = game.headers
    white = headers.get('White', '')
    black = headers.get('Black', '')
    result = headers.get('Result', '*')
    time_control = headers.get('TimeControl', '')
    utc_dt = datetime.strptime(
        headers['UTCDate'] + ' ' + headers['UTCTime'], '%Y.%m.%d %H:%M:%S'
    )
    rows.append({
        'event': headers.get('Event', ''),
        'white': white,
        'black': black,
        'opponent': black if white == username else white,
        'color': 'white' if white == username else 'black',
        'result': my_result(white, black, result),
        'time_control': time_control,
        'eco': headers.get('ECO', ''),
        'termination': headers.get('Termination', ''),
        'utc_datetime': utc_dt,
        'ekb_datetime': utc_dt + timedelta(hours=5),
    })

df = pd.DataFrame(rows)
df['ekb_date'] = df['ekb_datetime'].dt.date
df['year'] = df['ekb_datetime'].dt.year
df['month'] = df['ekb_datetime'].dt.month
df['day'] = df['ekb_datetime'].dt.day
df['weekday'] = df['ekb_datetime'].dt.weekday
df['hour'] = df['ekb_datetime'].dt.hour

assert len(df) == len(games)

other_count = (df['result'] == 'other').sum()
if other_count:
    print(f'Партий без результата (аборты и т.п.): {other_count}')

## Анализ хедеров

### Топ-10 событий по числу партий

In [ ]:
top_events = df['event'].value_counts().head(10)

plt.figure(figsize=(12, 4))
plt.bar(top_events.index, top_events.values)
plt.title('Топ-10 событий по числу партий')
plt.xlabel('Событие')
plt.ylabel('Число партий')
plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()

### Топ-10 соперников по числу партий

In [ ]:
top_opponents = df['opponent'].value_counts().head(10)

plt.figure(figsize=(12, 4))
plt.bar(top_opponents.index, top_opponents.values)
plt.title('Топ-10 соперников по числу партий')
plt.xlabel('Соперник')
plt.ylabel('Число партий')
plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()

### Распределение партий по результатам

In [ ]:
results_counts = df['result'].value_counts().reindex(RESULT_ORDER, fill_value=0)
results_counts = results_counts[results_counts > 0]

plt.figure(figsize=(12, 4))
plt.bar(
    [RESULT_LABELS[r] for r in results_counts.index],
    results_counts.values,
    color=[RESULT_COLORS[r] for r in results_counts.index],
)
plt.title('Результаты партий')
plt.xlabel('Результат')
plt.ylabel('Число партий')
plt.grid(axis='y', alpha=0.3, linestyle='--')

## Анализ даты и времени партий

#### Распределение партий по годам

In [ ]:
years_counts = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 4))
plt.bar(years_counts.index, years_counts.values)
plt.title('Распределение партий по годам')
plt.xlabel('Год')
plt.ylabel('Число партий')
plt.xticks(years_counts.index)
plt.grid(axis='y', alpha=0.3, linestyle='--')

#### Распределение партий по месяцам

In [ ]:
months_counts = df['month'].value_counts().reindex(range(1, 13), fill_value=0)

plt.figure(figsize=(12, 4))
plt.bar(months_counts.index, months_counts.values)
plt.title('Распределение партий по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Число партий')
plt.xticks(range(1, 13))
plt.grid(axis='y', alpha=0.3, linestyle='--')

#### Распределение партий по числам месяца

In [ ]:
days_counts = df['day'].value_counts().reindex(range(1, 32), fill_value=0)

plt.figure(figsize=(12, 4))
plt.bar(days_counts.index, days_counts.values)
plt.title('Распределение партий по числам месяца')
plt.xlabel('Число месяца')
plt.ylabel('Число партий')
plt.xticks(range(1, 32))
plt.grid(axis='y', alpha=0.3, linestyle='--')

#### Распределение партий по дням недели

In [ ]:
weekdays_counts = df['weekday'].value_counts().reindex(range(7), fill_value=0)

plt.figure(figsize=(12, 4))
plt.bar(WEEKDAY_NAMES_SHORT_RU, weekdays_counts.values)
plt.title('Распределение партий по дням недели')
plt.xlabel('День недели')
plt.ylabel('Число партий')
plt.grid(axis='y', alpha=0.3, linestyle='--')

#### Распределение партий по времени суток

In [ ]:
hours_counts = df['hour'].value_counts().reindex(range(24), fill_value=0)

plt.figure(figsize=(12, 4))
plt.bar(hours_counts.index, hours_counts.values)
plt.title('Распределение партий по времени суток')
plt.xlabel('Час')
plt.ylabel('Число партий')
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.3, linestyle='--')

#### Максимальное число партий, сыгранных за день

In [ ]:
games_per_day = df['ekb_date'].value_counts()
max_games_per_day = games_per_day.max()

print(f'Максимальное число партий: {max_games_per_day}\nБыло сыграно:')
for d, n in games_per_day.items():
    if n == max_games_per_day:
        print(f'{d} ({WEEKDAY_NAMES_FULL_RU[d.weekday()]})')

#### Самые длинные серии партий (каждый день серии сыграна хотя бы одна партия)

In [ ]:
def compute_streaks(games_per_day: pd.Series) -> dict:
    dates = sorted(games_per_day.index)
    streaks = {}

    streak_start = dates[0]
    streak_days = 1
    streak_games = games_per_day[dates[0]]

    for prev_date, date in zip(dates, dates[1:]):
        if date - prev_date == timedelta(days=1):
            streak_days += 1
            streak_games += games_per_day[date]
        else:
            streaks[streak_start] = (streak_days, streak_games)
            streak_start = date
            streak_days = 1
            streak_games = games_per_day[date]

    streaks[streak_start] = (streak_days, streak_games)  # не забыть последнюю (текущую) серию
    return streaks


streaks = compute_streaks(games_per_day)
max_days = max(days for days, _ in streaks.values())
max_games = max(games_count for _, games_count in streaks.values())

print('Самые длинные серии по числу дней:')
for start, (days, games_count) in streaks.items():
    if days == max_days:
        print(f'{start}: дни {days}, партии {games_count}')

print()

print('Самые длинные серии по числу партий:')
for start, (days, games_count) in streaks.items():
    if games_count == max_games:
        print(f'{start}: дни {days}, партии {games_count}')

### Распределение партий по контролям

In [ ]:
time_controls_counts = df['time_control'].value_counts()

plt.figure(figsize=(12, 4))
plt.bar(time_controls_counts.index, time_controls_counts.values)
plt.title('Распределение партий по контролям')
plt.xlabel('Контроль')
plt.ylabel('Число партий')
plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()

### Самые частые дебюты

In [ ]:
top_openings = df['eco'].value_counts().head(30)

plt.figure(figsize=(12, 5))
plt.bar(top_openings.index, top_openings.values)
plt.title('Самые частые дебюты')
plt.xlabel('Дебют')
plt.ylabel('Число партий')
plt.xticks(rotation=90)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()

### Распределение партий по типам завершений

In [ ]:
terminations_counts = df['termination'].value_counts()

plt.figure(figsize=(12, 4))
plt.bar(terminations_counts.index, terminations_counts.values)
plt.title('Распределение партий по типам завершений')
plt.xlabel('Тип завершения')
plt.ylabel('Число партий')
plt.grid(axis='y', alpha=0.3, linestyle='--')